# 📝 벡터 검색 과제 LV2 정답 — 상품 설명 검색 파이프라인 (강사용)

각 문제의 **모범답안 + 해설** 입니다. 경로는 `../../day15_RAG_벡터검색/data/` 입니다. 검색 결과는 하드웨어에 따라 미세하게 흔들릴 수 있어 **기대 문서 포함 여부·필터 조건 충족**으로 느슨하게 채점합니다.

아래 셀을 먼저 실행해 라이브러리와 한국어 임베딩 모델을 준비하세요.

In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한국어 임베딩 모델을 준비합니다.
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb

# 지난 단원에서 배운 한국어 임베딩 모델 — 문장 한 개를 768차원 벡터로 바꿉니다.
# (처음 부를 때 모델을 내려받느라 조금 걸릴 수 있어요.)
emb_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
print('임베딩 모델 준비 완료 — 벡터 차원:', emb_model.get_embedding_dimension())

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
(아래 셀은 실행만 하면 됩니다.)

In [ ]:
# [제공 코드] 상품 데이터를 살펴봅니다.
products = pd.read_csv('../../day15_RAG_벡터검색/data/products.csv')
print("행·열 크기:", products.shape)
print("\n[분류별 개수]"); print(products["category"].value_counts())
print("\n[가격 요약]"); print(products["price"].describe().round(0))
print("\n[앞 5행]"); display(products.head())

## 1. 색인 파이프라인 — 임베딩부터 적재까지 한 번에
**배경**: 검색을 하려면 먼저 문서를 **임베딩해 컬렉션에 넣어야**(인덱싱) 합니다. 이 과정을 한 셀에 이어서 만들어 봅니다.

**요구사항**:
- `products['description']` 전체를 임베딩해 변수 `prod_emb` 에 담으세요(`normalize_embeddings=True`).
- `chromadb.EphemeralClient()` 로 클라이언트를 열고, `get_or_create_collection('shop_items', metadata={'hnsw:space': 'cosine'})` 로 컬렉션을 만들어 변수 `shop_items` 에 담으세요.
- `shop_items.add(...)` 로 21개를 넣으세요 — `ids`, `embeddings=prod_emb`, `documents`, 그리고 `metadatas` 는 각 상품의 `{'name':…, 'category':…, 'price': int(가격)}` 리스트.
- `shop_items.count()` 가 `21` 인지 확인하세요.

**예시**
```
prod_emb.shape      →  (21, 768)
shop_items.count()  →  21
```
<details><summary>힌트</summary>

```text
접근방법:
- 설명 열 전체를 임베딩하고, 코사인 컬렉션을 만든 뒤, id·임베딩·원문·메타(가격 포함)를 add 한다.

세부구현:
1. description 을 리스트로 만들어 encode 로 prod_emb 에 담는다
2. EphemeralClient 로 client 를, get_or_create_collection 으로 shop_items 를 만든다
3. add 에 ids·embeddings·documents·metadatas 를 넘긴다(price 는 int 로)
4. count 로 21 을 확인한다
```

</details>

In [ ]:
prod_emb = emb_model.encode(products['description'].tolist(), normalize_embeddings=True)
client = chromadb.EphemeralClient()
shop_items = client.get_or_create_collection('shop_items', metadata={'hnsw:space': 'cosine'})
shop_items.add(
    ids=products['id'].tolist(),
    embeddings=prod_emb,
    documents=products['description'].tolist(),
    metadatas=[{'name': n, 'category': c, 'price': int(p)}
               for n, c, p in zip(products['name'], products['category'], products['price'])],
)
print("상품 임베딩:", prod_emb.shape, "/ 저장된 문서 수:", shop_items.count())

In [ ]:
# [자가채점]
assert prod_emb.shape == (21, 768)
assert shop_items.count() == 21
print("✅ 문제1 통과!")

### 해설 — 문제 1
- **접근법**: '임베딩 → 컬렉션 생성 → add' 가 인덱싱 타임의 전부입니다. 가격을 메타데이터에 **숫자(int)** 로 넣어 두면 뒤에서 **숫자 조건 필터**를 걸 수 있습니다.
- **흔한 실수**: `price` 를 문자열로 넣으면 `$lt` 같은 숫자 비교 필터가 동작하지 않습니다. `int(p)` 로 변환해 넣으세요.

아래는 이후 문제(2~6)에서 쓸 `shop_items` 컬렉션을 **확실히 준비**하는 제공 코드입니다. (문제 1을 풀었다면 그대로 유지됩니다.)

In [ ]:
# [제공 코드] 이후 검색 문제에서 쓸 컬렉션(shop_items)을 확실히 준비합니다.
# (문제 1을 이미 풀었다면 그대로 유지되고, 건너뛰었더라도 여기서 채워집니다.)
products = pd.read_csv('../../day15_RAG_벡터검색/data/products.csv')
prod_emb = emb_model.encode(products['description'].tolist(), normalize_embeddings=True)
client = chromadb.EphemeralClient()
shop_items = client.get_or_create_collection('shop_items', metadata={'hnsw:space': 'cosine'})
shop_items.add(
    ids=products['id'].tolist(),
    embeddings=prod_emb,
    documents=products['description'].tolist(),
    metadatas=[{'name': n, 'category': c, 'price': int(p)}
               for n, c, p in zip(products['name'], products['category'], products['price'])],
)
print('shop_items 문서 수:', shop_items.count())

## 2. 의미 검색 — 등산 장비 찾기
**배경**: 이제 자연어로 상품을 검색합니다.

**요구사항**:
- 질문 `"산에 오를 때 필요한 장비"` 를 임베딩해 `shop_items.query(..., n_results=3)` 로 검색하고, 문서 id 리스트를 `hit_ids` 에 담으세요.
- 이 질문에는 등산 배낭(id `p01`)이 가장 잘 맞습니다. `'p01'` 이 `hit_ids` 에 있는지 확인합니다.

**예시**
```
'p01' in hit_ids  →  True
```
<details><summary>힌트</summary>

```text
접근방법:
- 질문을 임베딩해 query_embeddings 로 넣고 n_results=3 으로 검색한 뒤 ids[0] 을 꺼낸다.

세부구현:
1. 질문을 임베딩한다
2. shop_items.query 로 n_results=3 검색한다
3. 결과의 ids[0] 을 hit_ids 에 담는다
```

</details>

In [ ]:
q_emb = emb_model.encode(["산에 오를 때 필요한 장비"], normalize_embeddings=True)
res = shop_items.query(query_embeddings=q_emb, n_results=3)
hit_ids = res['ids'][0]
for meta in res['metadatas'][0]:
    print(f"  {meta['name']} ({meta['category']}, {meta['price']}원)")

In [ ]:
# [자가채점]
assert len(hit_ids) == 3, 'n_results=3 으로 검색해야 합니다'
assert 'p01' in hit_ids
print("✅ 문제2 통과!")

### 해설 — 문제 2
- **접근법**: '산에 오를 때 필요한 장비'는 등산용품 설명과 의미가 가깝습니다. 상위 결과의 `category` 가 대부분 '등산용품'으로 나옵니다.

## 3. 메타데이터 필터 — 전자기기 안에서만
**배경**: 의미 검색에 **분류 조건**을 함께 겁니다.

**요구사항**:
- 질문 `"선물하기 좋은 제품"` 을 임베딩해 `shop_items.query(..., n_results=3, where={'category': '전자기기'})` 로 검색하고, 결과 메타데이터의 분류 리스트를 `cats` 에 담으세요.
- `cats` 가 **모두 '전자기기'** 인지 확인합니다.

**예시**
```
cats  →  ['전자기기', '전자기기', '전자기기']
```
<details><summary>힌트</summary>

```text
접근방법:
- query 에 where={'category': '전자기기'} 를 함께 넘기고, 반환 메타에서 category 만 뽑아 확인한다.

세부구현:
1. 질문을 임베딩한다
2. shop_items.query 에 n_results=3, where={'category': '전자기기'} 를 넘긴다
3. metadatas[0] 의 각 원소에서 category 를 뽑아 cats 리스트를 만든다
```

</details>

In [ ]:
q_emb = emb_model.encode(["선물하기 좋은 제품"], normalize_embeddings=True)
res = shop_items.query(query_embeddings=q_emb, n_results=3, where={'category': '전자기기'})
cats = [m['category'] for m in res['metadatas'][0]]
print("걸러진 분류:", cats)

In [ ]:
# [자가채점]
assert len(cats) == 3, '전자기기는 3개라 n_results=3 이면 3개가 모두 나옵니다'
assert all(c == '전자기기' for c in cats)
print("✅ 문제3 통과!")

### 해설 — 문제 3
- **접근법**: `where` 로 분류를 고정하면, 의미가 가까워도 다른 분류는 걸러집니다.

## 4. 숫자 필터 — 3만 원 미만 상품만
**배경**: 메타데이터가 숫자이면 **크기 조건**을 걸 수 있습니다. 값을 그대로 주면 '정확히 그 값'을 뜻하고, 크기 조건은 **연산자 딕셔너리**로 감싸 줍니다 — `$lt`(미만)·`$lte`(이하)·`$gt`(초과)·`$gte`(이상). 가격이 **3만 원 미만**인 상품만 검색해 봅시다.

**요구사항**:
- 질문 `"집에서 쓰기 좋은 저렴한 생활용품"` 을 임베딩해 `shop_items.query(..., n_results=3)` 로 검색하되, **가격이 30000 미만**인 상품만 나오도록 `where` 조건을 함께 거세요. 결과 메타의 가격 리스트를 `prices` 에 담습니다.
- `prices` 가 **모두 30000 미만** 인지 확인합니다.

**예시**
```
prices  →  [9000, 24000, 27000]   (모두 30000 미만)
```
<details><summary>힌트</summary>

```text
접근방법:
- where 의 price 에 '미만' 연산자를 딕셔너리로 감싸 넣는다. 반환 메타에서 price 를 뽑아 확인한다.

세부구현:
1. 질문을 임베딩한다
2. shop_items.query 에 n_results=3 과 함께, price 에 $lt 로 30000 미만 조건을 건 where 를 넘긴다
3. metadatas[0] 의 각 원소에서 price 를 뽑아 prices 리스트를 만든다
```

</details>

In [ ]:
q_emb = emb_model.encode(["집에서 쓰기 좋은 저렴한 생활용품"], normalize_embeddings=True)
res = shop_items.query(query_embeddings=q_emb, n_results=3, where={'price': {'$lt': 30000}})
prices = [m['price'] for m in res['metadatas'][0]]
print("걸러진 가격:", prices)

In [ ]:
# [자가채점]
assert len(prices) == 3, '3만 원 미만 상품은 8개라 n_results=3 이면 3개가 나옵니다'
assert all(p < 30000 for p in prices)
print("✅ 문제4 통과!")

### 해설 — 문제 4
- **접근법**: 숫자 메타데이터에는 `$lt`(미만)·`$lte`(이하)·`$gt`(초과)·`$gte`(이상) 같은 연산자를 쓸 수 있습니다. 문제 1 에서 가격을 **숫자로** 넣어 뒀기에 이 필터가 동작합니다.
- **흔한 실수**: `where={'price': 30000}` 은 '가격이 정확히 30000' 을 뜻합니다. **미만**은 반드시 `{'$lt': 30000}` 처럼 연산자로 감싸야 합니다.

## 5. 조합 — 의미 검색 + 분류 필터
**배경**: 의미 검색과 분류 필터를 **함께** 걸어, '이 분류 중에서 가장 관련 있는 상품'을 찾습니다.

**요구사항**:
- 질문 `"겨울에 따뜻하게 입을 옷"` 을 임베딩해 `shop_items.query(..., n_results=3, where={'category': '의류'})` 로 검색하고, 결과 id 리스트를 `warm_ids`, 분류 리스트를 `warm_cats` 에 담으세요.
- `warm_cats` 가 **모두 '의류'** 이고, 경량 패딩(id `p08`)이 `warm_ids` 에 있는지 확인합니다.

**예시**
```
warm_cats         →  ['의류', '의류', '의류']
'p08' in warm_ids →  True
```
<details><summary>힌트</summary>

```text
접근방법:
- query 에 where={'category': '의류'} 를 걸어 검색하고, ids 와 category 를 각각 꺼낸다.

세부구현:
1. 질문을 임베딩한다
2. shop_items.query 에 n_results=3, where={'category': '의류'} 를 넘겨 res 에 담는다
3. res['ids'][0] 을 warm_ids, 메타의 category 를 warm_cats 에 담는다
```

</details>

In [ ]:
q_emb = emb_model.encode(["겨울에 따뜻하게 입을 옷"], normalize_embeddings=True)
res = shop_items.query(query_embeddings=q_emb, n_results=3, where={'category': '의류'})
warm_ids = res['ids'][0]
warm_cats = [m['category'] for m in res['metadatas'][0]]
for meta in res['metadatas'][0]:
    print(f"  {meta['name']} ({meta['category']}, {meta['price']}원)")

In [ ]:
# [자가채점]
assert len(warm_cats) == 3, '의류는 3개라 n_results=3 이면 3개가 모두 나옵니다'
assert all(c == '의류' for c in warm_cats)
assert 'p08' in warm_ids
print("✅ 문제5 통과!")

### 해설 — 문제 5
- **접근법**: 의미 검색(따뜻한 옷)과 분류 필터(의류)를 함께 걸면, 의류 중에서 겨울옷이 위로 올라옵니다. 경량 패딩이 대표적으로 잡힙니다.
- **대안**: 분류 대신 가격 조건을 함께 걸거나, 두 조건을 `$and` 로 묶을 수도 있습니다.

## 6. 숫자 필터 — 10만 원 이상 고가 상품만
**배경**: 문제 4 의 반대 방향으로, **비싼 상품**만 걸러 봅니다. `$gte` 는 '이상'입니다.

**요구사항**:
- 질문 `"성능 좋은 고급 제품"` 을 임베딩해 `shop_items.query(..., n_results=3)` 로 검색하되, **가격이 100000 이상**인 상품만 나오도록 `where` 조건을 거세요. 결과 가격 리스트를 `hi_prices` 에 담습니다.
- `hi_prices` 가 **모두 100000 이상** 인지 확인합니다.

**예시**
```
hi_prices  →  [139000, 129000, 119000]   (모두 100000 이상)
```
<details><summary>힌트</summary>

```text
접근방법:
- 문제 4 와 같되 연산자를 $gte(이상), 기준을 100000 으로 바꾼다.

세부구현:
1. 질문을 임베딩한다
2. shop_items.query 에 n_results=3 과 함께, price 에 $gte 로 100000 이상 조건을 건 where 를 넘긴다
3. 메타에서 price 를 뽑아 hi_prices 에 담는다
```

</details>

In [ ]:
q_emb = emb_model.encode(["성능 좋은 고급 제품"], normalize_embeddings=True)
res = shop_items.query(query_embeddings=q_emb, n_results=3, where={'price': {'$gte': 100000}})
hi_prices = [m['price'] for m in res['metadatas'][0]]
print("걸러진 가격:", hi_prices)

In [ ]:
# [자가채점]
assert len(hi_prices) == 3, '10만 원 이상 상품은 3개라 n_results=3 이면 3개가 모두 나옵니다'
assert all(p >= 100000 for p in hi_prices)
print("✅ 문제6 통과!")

### 해설 — 문제 6
- **접근법**: `$lt`↔`$gte` 처럼 연산자만 바꾸면 조건 방향이 바뀝니다. 숫자 메타데이터 하나로 가격대별 검색을 자유롭게 만들 수 있습니다.
- **대안**: `{'$gte': 50000, '$lt': 100000}` 처럼 상·하한을 함께 주면 **가격 구간** 검색이 됩니다.

## 7. 여러 질문을 한 번에 검색하기
**배경**: 실제 검색 서비스에는 질문이 **여러 개** 들어옵니다. 질문 리스트를 **반복문으로 돌며** 각 질문에 가장 관련 있는 상품 1개씩을 모아 봅니다(검색과 반복의 조합).

**요구사항**:
- 아래 세 질문을 순서대로 검색해, 각 질문의 **Top-3 상품 id 리스트**를 `hits_per_q` 에 모으고, 거기서 **1위만** 뽑은 리스트를 `top_ids` 에 담으세요.
```
questions = ['겨울 등산에 신는 방수 신발', '요리할 때 쓰는 프라이팬', '캠핑장에서 앉을 의자']
```
- 각 질문을 임베딩해 `shop_items.query(..., n_results=3)` 로 검색하고, 그 질문의 결과 id 리스트(`res['ids'][0]`)를 `hits_per_q` 에 하나씩 붙입니다.
- `top_ids` 는 각 Top-3 의 **첫 원소**만 모은 리스트입니다.
- 첫 질문의 1위는 `'p02'`(방수 트레킹화), 세 번째 질문의 1위는 `'p19'`(캠핑용 접이식 의자) 이고, 두 번째 질문의 Top-3 에는 `'p11'`(논스틱 프라이팬) 이 들어옵니다.

**예시**
```
hits_per_q[0]  →  ['p02', 'p09', 'p08']   (첫 질문의 Top-3)
top_ids        →  ['p02', 'p11', 'p19']   (각 질문의 1위)
```
<details><summary>힌트</summary>

```text
접근방법:
- 질문 리스트를 for 로 돌며, 각 질문의 Top-3 id 리스트를 모으고 마지막에 1위만 따로 뽑는다.

세부구현:
1. hits_per_q 빈 리스트를 만든다.
2. questions 를 for 로 돌며 각 질문을 임베딩해 n_results=3 으로 검색한다.
3. 그 질문의 결과 id 리스트(ids[0])를 hits_per_q 에 append 한다.
4. hits_per_q 의 각 원소에서 첫 id 만 뽑아 top_ids 를 만든다.
```

</details>

In [ ]:
questions = ['겨울 등산에 신는 방수 신발', '요리할 때 쓰는 프라이팬', '캠핑장에서 앉을 의자']
hits_per_q = []
for q in questions:
    q_emb = emb_model.encode([q], normalize_embeddings=True)
    res = shop_items.query(query_embeddings=q_emb, n_results=3)
    hits_per_q.append(res['ids'][0])
top_ids = [hits[0] for hits in hits_per_q]
for q, hits in zip(questions, hits_per_q):
    print(f'{q} -> {hits}')
print('질문별 top-1:', top_ids)

In [ ]:
# [자가채점]
assert len(hits_per_q) == 3 and all(len(h) == 3 for h in hits_per_q), \
    '질문 3개를 각각 n_results=3 으로 검색해야 합니다'
assert top_ids == [h[0] for h in hits_per_q], 'top_ids 는 각 Top-3 의 1위를 모은 것이어야 합니다'
assert top_ids[0] == 'p02', '첫 질문(방수 신발)의 1위는 p02 입니다'
assert top_ids[2] == 'p19', '세 번째 질문(캠핑 의자)의 1위는 p19 입니다'
assert 'p11' in hits_per_q[1], '두 번째 질문(프라이팬)의 Top-3 에 p11 이 들어와야 합니다'
print("✅ 문제7 통과!")

### 해설 — 문제 7
- **접근법**: 검색 한 번을 반복문에 넣으면 '질문 여러 개 → 결과 여러 개' 가 됩니다. 각 질문이 서로 다른 분류의 상품을 정확히 찾아내는 것을 볼 수 있습니다.
- **흔한 실수**: `query` 반환의 `ids` 는 **2차원**입니다(`ids[0]` 이 첫(그리고 유일한) 질문의 결과 리스트). top-1 만 필요하면 `ids[0][0]` 처럼 두 번 인덱싱합니다.
- **왜 Top-3 까지 보나**: 프라이팬 질문은 1위(`p11`)와 2위(`p21` 휴대용 가스 버너)의 거리 차가 작습니다. 이렇게 후보가 붙어 있을 때는 1위만 보지 말고 **상위 몇 개를 함께** 보는 편이 안전합니다 — RAG 에서 K 를 3~5 로 두는 이유이기도 합니다.

## 8. Top-K 비교 — K 를 바꾸면 결과가 몇 개?
**배경**: 검색에서 `n_results`(K)는 **몇 개를 가져올지** 를 정합니다. 같은 질문을 K=1 과 K=5 로 검색해 결과 수와 1위가 어떻게 되는지 비교합니다(검색과 K 변화의 조합).

**요구사항**:
- 질문 `"따뜻하게 입는 겨울 옷"` 을 임베딩해 **두 번** 검색하세요 — 한 번은 `n_results=1`, 한 번은 `n_results=5`.
- 각 결과의 id 리스트를 `ids_1`, `ids_5` 에 담으세요.
- `ids_1` 은 1개, `ids_5` 는 5개이고, **1위(첫 결과)가 서로 같은지**(`ids_1[0] == ids_5[0]`) 확인합니다. 이 질문의 1위는 경량 패딩 점퍼(id `p08`)입니다.

**예시**
```
len(ids_1)           →  1
len(ids_5)           →  5
ids_1[0]             →  'p08'
ids_1[0] == ids_5[0] →  True   (K 를 늘려도 1위는 그대로, 아래 순위가 더 채워짐)
```
<details><summary>힌트</summary>

```text
접근방법:
- 같은 질문 임베딩으로 n_results 만 1·5 로 바꿔 두 번 검색하고, 각 결과의 ids[0] 을 꺼낸다.

세부구현:
1. 질문을 한 번 임베딩한다.
2. n_results=1 로 검색해 ids[0] 을 ids_1 에, n_results=5 로 검색해 ids[0] 을 ids_5 에 담는다.
3. 길이와 첫 원소를 비교한다.
```

</details>

In [ ]:
q_emb = emb_model.encode(["따뜻하게 입는 겨울 옷"], normalize_embeddings=True)
ids_1 = shop_items.query(query_embeddings=q_emb, n_results=1)['ids'][0]
ids_5 = shop_items.query(query_embeddings=q_emb, n_results=5)['ids'][0]
print('K=1:', ids_1)
print('K=5:', ids_5)

In [ ]:
# [자가채점]
assert len(ids_1) == 1
assert len(ids_5) == 5
assert ids_1[0] == 'p08', \
    "'따뜻하게 입는 겨울 옷' 의 1위는 경량 패딩 점퍼(p08) 입니다 - 지문의 질문으로 검색했는지 확인하세요"
assert ids_1[0] == ids_5[0], 'K 를 바꿔도 1위는 같아야 합니다'
print("✅ 문제8 통과!")

### 해설 — 문제 8
- **접근법**: `n_results` 는 **재현율과 속도의 트레이드오프**입니다. K 를 늘리면 관련 문서를 놓칠 확률은 줄지만, 관련 낮은 것도 함께 섞입니다. 1위는 그대로이고 아래 순위가 더 채워집니다.
- **대안**: RAG 에서는 보통 K=3~5 로 시작해, 검색 품질 지표(다음 단원)를 보며 조정합니다.

## 9. 같은 검색을 Qdrant 로 재현하기 (조합)
**배경**: 교안 마지막에서 **같은 검색을 ChromaDB 와 Qdrant 로 각각** 해 봤습니다. 벡터 DB 는 도구가 달라도 **적재 → 질문 임베딩 → Top-K** 라는 뼈대가 같다는 것을 직접 확인합니다. 이번엔 Qdrant 로 위와 같은 상품 검색을 재현하세요.

**요구사항**:
- `QdrantClient(':memory:')` 로 클라이언트를 만들어 `qclient` 에 담으세요(설치·서버 없이 메모리에서 동작).
- 컬렉션 이름은 `'shop_qdrant'`, 벡터 설정은 **`prod_emb` 의 차원**과 **코사인 거리**로 만드세요.
- 상품 21개를 `PointStruct` 로 만들어 `upsert` 하세요 — `id` 는 0부터의 정수, `vector` 는 `prod_emb[i]`, `payload` 에는 `{'name': ..., 'category': ...}` 를 넣습니다.
- 질문 **'등산 갈 때 멜 배낭'** 을 임베딩해 `query_points(..., limit=3)` 로 검색하고, 결과 점들의 `payload['category']` 를 순서대로 리스트 `qdrant_cats` 에 담으세요.

**예시**
```
qclient.count('shop_qdrant').count  ->  21
qdrant_cats                         ->  상위 3개 상품의 분류 3개 (리스트)
```
<details><summary>힌트</summary>

```text
접근방법:
- 교안의 Qdrant 절과 같은 순서다: 클라이언트 -> 컬렉션 생성 -> upsert -> query_points.

세부구현:
1. qdrant_client 에서 QdrantClient 를, qdrant_client.models 에서 Distance·VectorParams·PointStruct 를 가져온다
2. 메모리 클라이언트를 만들고, 벡터 차원(prod_emb.shape[1])과 코사인 거리로 컬렉션을 만든다
3. 상품마다 PointStruct 를 만들어 한 번에 upsert 한다 (vector 는 리스트로 변환해야 한다)
4. 질문을 emb_model 로 임베딩해 query_points 에 넘기고, 반환된 points 의 payload 에서 분류를 모은다
```

</details>

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

qclient = QdrantClient(':memory:')
dim = prod_emb.shape[1]
if qclient.collection_exists('shop_qdrant'):
    qclient.delete_collection('shop_qdrant')
qclient.create_collection('shop_qdrant',
                          vectors_config=VectorParams(size=dim, distance=Distance.COSINE))

qclient.upsert('shop_qdrant', points=[
    PointStruct(id=i, vector=prod_emb[i],
                payload={'name': products.loc[i, 'name'],
                         'category': products.loc[i, 'category']})
    for i in range(len(products))
])
print('Qdrant 저장 점 개수:', qclient.count('shop_qdrant').count)

q_emb = emb_model.encode(['등산 갈 때 멜 배낭'], normalize_embeddings=True)
hits = qclient.query_points('shop_qdrant', query=q_emb[0], limit=3).points
qdrant_cats = [h.payload['category'] for h in hits]
for h in hits:
    print(f"  점수 {h.score:.3f}  |  {h.payload['name']} ({h.payload['category']})")
print('분류:', qdrant_cats)

In [ ]:
# [자가채점]
assert qclient.count('shop_qdrant').count == 21
assert isinstance(qdrant_cats, list) and len(qdrant_cats) == 3
# 컬렉션을 코사인 거리로 만들었는지 — 설정 자체를 확인한다
from qdrant_client.models import Distance as _Distance
assert qclient.get_collection('shop_qdrant').config.params.vectors.distance == _Distance.COSINE, \
    '컬렉션의 거리 척도를 코사인(Distance.COSINE)으로 만들어야 합니다'
# 도구가 달라도 결과는 같아야 한다 — 같은 질문의 1위가 '같은 상품'인지 id 수준으로 대조
_qe = emb_model.encode(['등산 갈 때 멜 배낭'], normalize_embeddings=True)
_chroma_top_id = shop_items.query(query_embeddings=_qe, n_results=1)['ids'][0][0]
_qtop = qclient.query_points('shop_qdrant', query=_qe[0], limit=1).points[0]
_qdrant_top_id = products.loc[_qtop.id, 'id']   # point id(0부터의 정수) → 상품 id
assert _qdrant_top_id == _chroma_top_id, \
    f'ChromaDB 1위({_chroma_top_id})와 Qdrant 1위({_qdrant_top_id})가 같은 상품이어야 합니다'
assert _qtop.payload['name'] == products.loc[_qtop.id, 'name'], \
    'point 의 id 와 payload 가 같은 상품을 가리켜야 합니다(순서가 어긋나지 않았는지)'
print("✅ 문제9 통과! (두 DB 의 1위 상품:", _chroma_top_id, ")")

### 해설 — 문제 9
- **접근법**: ChromaDB 와 이름만 다를 뿐 **뼈대가 같습니다** — 컬렉션을 만들고, 벡터와 메타데이터를 넣고, 질문 벡터로 Top-K 를 가져옵니다. 그래서 **같은 임베딩·같은 질문이면 1위도 같은 상품**입니다 — 자가채점이 두 DB 의 1위를 **상품 id 수준**으로 대조합니다(분류만 같은 것으로는 통과하지 못합니다). `id=0,1,2…` 를 행 순서대로 넣었기에 `products.loc[point.id, 'id']` 로 되돌릴 수 있습니다.
- **흔한 실수**: 컬렉션을 만들 때 `VectorParams(size=..., distance=...)` 의 **차원과 거리 척도**를 적재할 벡터와 맞추지 않는 것입니다. 768차원 벡터를 넣을 컬렉션을 512로 만들면 적재부터 막히고, 거리 척도가 다르면 검색 결과가 어긋납니다.
- **대안**: `QdrantClient(':memory:')` 대신 `QdrantClient(path='...')` 로 디스크에 저장하거나, 서버·클라우드에 붙일 수 있습니다 — 코드의 나머지는 그대로입니다. 이것이 '도구를 갈아 끼워도 뼈대는 같다'는 뜻입니다.

## 10. 무엇과 무엇을 맞바꾸나 (서술형)
**배경**: 지금까지 `query()` 한 줄로 검색했지만, 그 안에서는 **모든 문서와 비교하지 않습니다**. 교안에서 본 **ANN(근사 최근접 탐색)·HNSW** 가 후보를 추려 빠르게 찾습니다.

**요구사항**: 아래 두 질문에 각각 **2~3문장**으로 답하세요(코드 없음, 아래 마크다운 셀에 직접 작성).

1. **`K`(`n_results`)를 키우면** 무엇이 좋아지고 무엇이 나빠지나요?
2. **`ANN`은 무엇을 내주고 무엇을 얻나요?** 그리고 교안에서 인덱스 설정(`search_ef`·`M`)을 낮췄을 때 `recall` 이 어떻게 됐는지 함께 적으세요.

> 💡 둘은 **다른 층위의 맞바꿈**입니다. K 는 *몇 개를 가져올까*, ANN 은 *얼마나 꼼꼼히 찾을까* 의 문제입니다. 이 둘을 구분해 설명할 수 있으면 벡터 DB 설정을 스스로 고를 수 있습니다.

> 이 문제는 **자가채점이 없습니다** — 정답 노트북의 모범 서술과 비교해 보세요.

**모범 서술 (예시 — 표현은 달라도 됩니다)**

1. **K 를 키우면** 정답을 놓칠 확률이 줄어(재현율↑) 대신 관련 없는 문서가 함께 딸려 와 정밀도가 떨어지고, 생성 단계에 넣을 프롬프트가 길어져 **비용과 지연**도 늘어납니다. 즉 **재현율 ↔ 정밀도·비용**의 맞바꿈입니다.

2. **ANN 은 '정확히 가장 가까운 것'을 포기하는 대신 속도를 얻습니다.** 전부 비교(선형 스캔)하지 않고 미리 만든 그래프(HNSW)를 따라 유망한 후보만 살피기 때문입니다. 교안에서 탐색 폭(`search_ef`)과 그래프 연결 수(`M`)를 **함께 낮추자** 질의 시간은 짧아졌지만 **recall@10 이 1.00 에서 0.1 미만으로 뚝 떨어졌습니다** — 후보를 적게 볼수록 진짜 이웃을 놓친다는 뜻입니다. 즉 **속도 ↔ 정확도**의 맞바꿈이고, K 의 맞바꿈(몇 개를 가져올까)과는 층위가 다릅니다.

### 해설 — 문제 10
- **핵심**: 두 맞바꿈을 **섞지 않는 것**이 이 문제의 목적입니다. K 는 **검색 결과를 몇 개 쓸지**(사용 측), ANN 파라미터는 **인덱스를 얼마나 꼼꼼히 뒤질지**(엔진 측)입니다. 학생이 "K 를 줄이면 빨라진다"로만 이해하고 넘어가면, 나중에 검색이 느릴 때 만져야 할 손잡이를 못 찾습니다.
- **실무 연결**: 대개 **ANN 은 기본값을 그대로 두고**(실무에서 쓰기에 충분한 수준으로 잡혀 있습니다) **K 만 조절**합니다. ANN 손잡이는 문서가 수백만 건이 되어 지연이 문제될 때 비로소 건드립니다.

---
## 11. 검색 결과를 근거로 상품 추천 문장 쓰기

> ⚠️ **이 문제는 `.env` 의 `OPENAI_API_KEY` 가 필요합니다.** 앞의 문제들과 달리 **실제 API 를 호출**하므로 **요금이 조금 듭니다**(이 문제에서 호출 1회). 아래 제공 코드 셀을 먼저 실행하세요.

**배경**: 지금까지는 **찾기(Retrieval)** 까지만 했습니다. RAG 의 나머지 절반은 **찾은 것을 근거로 답을 쓰는 것(Generation)** 입니다. 검색 결과를 **컨텍스트 문자열**로 엮어 모델에게 건네고, **그 안에서만** 고르게 규칙을 주면 근거 있는 추천이 됩니다.

In [ ]:
# [제공 코드] 답 생성에 쓸 OpenAI 클라이언트를 준비합니다 — 이 셀을 먼저 실행하세요.
# .env 파일에 OPENAI_API_KEY 를 넣어 두면 아래 한 줄이 그것을 읽어 연결합니다.
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('.env')                # 같은 폴더의 .env
load_dotenv('../../day15_RAG_벡터검색/.env')             # (정답 폴더처럼 한 단계 안에서 열었을 때)

# 키가 없으면 OpenAI() 를 만드는 순간 알아보기 어려운 에러가 납니다.
# 그래서 먼저 확인하고, 없으면 무엇을 해야 하는지 알려 줍니다.
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError(
        '이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n'
        '  1) 일차 폴더에서  cp .env.example .env\n'
        '  2) .env 를 열어 OPENAI_API_KEY 에 본인 키를 채우기\n'
        '  3) 커널을 다시 시작한 뒤 이 셀부터 실행'
    )

# max_retries : 분당 한도에 걸려 거절당하면(429) 잠시 뒤 자동으로 다시 시도할 횟수
client = OpenAI(max_retries=8)     # OPENAI_API_KEY 를 환경변수에서 자동으로 찾아 쓴다
print('연결 준비 완료 — 키 확인됨')

**요구사항**:
- 질문 `"캠핑장에서 요리해 먹을 때 쓸 물건을 추천해 주세요"` 를 임베딩해 `shop_items.query(..., n_results=3)` 로 검색하고, 결과 메타데이터에서 **상품 이름 3개**를 `rec_names` 에 담으세요.
- 검색된 3개의 **이름·분류·가격**을 한 줄씩 이어 붙여 **컨텍스트 문자열** `context` 를 만드세요 (세 이름이 모두 들어가야 합니다).
- `messages` 리스트를 만드세요 — **`system`** 역할로 "주어진 상품 목록 안에서만 고르고, 이름은 목록에 적힌 그대로 쓰고, 마땅한 상품이 없으면 없다고 말하라" 는 규칙을 주고, **`user`** 역할로 `context` 와 질문을 함께 넣습니다.
- `client.chat.completions.create(model='gpt-4o-mini', messages=messages, temperature=0, max_tokens=200)` 로 호출해, 답변 문자열을 `answer` 에 담고 출력하세요.

**예시**
```
rec_names  →  ['휴대용 가스 버너', '캠핑용 접이식 의자', '논스틱 프라이팬 28cm']
context    →  '- 휴대용 가스 버너 (캠핑용품, 42000원)\n- 캠핑용 접이식 의자 (...)\n- ...'
answer     →  '캠핑장에서 요리하신다면 휴대용 가스 버너를 추천드려요. …' (매번 조금씩 달라집니다)
```
<details><summary>힌트</summary>

```text
접근방법:
- 검색 → 컨텍스트 문자열 만들기 → system 규칙과 함께 모델에 넘기기, 세 단계다.

세부구현:
1. 질문을 임베딩해 n_results=3 으로 검색하고, metadatas[0] 을 꺼낸다
2. 각 메타에서 name 을 모아 rec_names 를 만든다
3. 이름·분류·가격을 f-string 한 줄로 만들고 줄바꿈으로 이어 붙여 context 를 만든다
4. system 규칙 한 개와 user 메시지 한 개로 messages 리스트를 만든다 (user 에는 context 와 질문을 같이 넣는다)
5. chat.completions.create 를 호출하고 choices[0].message.content 를 answer 에 담는다
```

</details>

In [ ]:
rec_q = "캠핑장에서 요리해 먹을 때 쓸 물건을 추천해 주세요"
q_emb = emb_model.encode([rec_q], normalize_embeddings=True)
res = shop_items.query(query_embeddings=q_emb, n_results=3)
rec_metas = res['metadatas'][0]
rec_names = [m['name'] for m in rec_metas]
context = '\n'.join(f"- {m['name']} ({m['category']}, {m['price']}원)" for m in rec_metas)
print('[컨텍스트]')
print(context)

messages = [
    {'role': 'system',
     'content': '너는 쇼핑몰 상담원이다. 아래 상품 목록 안에서만 골라 추천하고, 상품 이름은 '
                '목록에 적힌 그대로 쓴다. 목록에 마땅한 상품이 없으면 없다고 말한다.'},
    {'role': 'user',
     'content': f'[상품 목록]\n{context}\n\n[질문] {rec_q}\n한두 문장으로 추천해 줘.'},
]
answer = client.chat.completions.create(
    model='gpt-4o-mini', messages=messages, temperature=0, max_tokens=200
).choices[0].message.content
print('\n[생성된 추천]')
print(answer)

In [ ]:
# [자가채점]  LLM 답변은 매번 조금씩 달라지므로 '정확히 이 문장' 으로는 채점할 수 없습니다.
#            그래서 1) 컨텍스트가 제대로 만들어졌는지 2) 답변이 실제로 왔는지
#            3) 근거(검색 결과) 안에서 답했는지 4) system 규칙을 주었는지 를 봅니다.
assert len(rec_names) == 3, 'n_results=3 으로 검색한 상품 이름 3개를 rec_names 에 담으세요'
assert all(_n in context for _n in rec_names), \
    '컨텍스트 문자열에 검색된 상품 3개의 이름이 모두 들어 있어야 합니다'
assert isinstance(answer, str) and len(answer.strip()) >= 20, \
    '답변이 비어 있거나 너무 짧습니다 - API 호출이 성공했는지 확인하세요'
assert any(_n in answer for _n in rec_names), \
    '답변에 검색된 상품 이름이 하나도 없습니다 - 목록 밖의 상품을 지어내지 않도록 '\
    'system 규칙에 "이름은 목록에 적힌 그대로" 를 넣고 다시 실행해 보세요'
assert any(_m.get('role') == 'system' for _m in messages), \
    'messages 에 system 역할로 규칙을 주어야 합니다'
print("✅ 문제11 통과!")

### 해설 — 문제 11
- **접근법**: RAG 의 마지막 조각입니다. 검색 결과를 **문자열 하나로 엮어** 프롬프트에 넣고, `system` 으로 **행동 규칙**(목록 밖은 고르지 말 것)을 못 박습니다. 모델은 자기가 아는 상품이 아니라 **우리가 준 목록**을 근거로 답하게 됩니다.
- **왜 정확 일치로 채점하지 않나**: LLM 출력은 같은 입력이라도 **매번 표현이 달라집니다** (`temperature=0` 이어도 완전히 같다고 보장되지 않습니다). 그래서 문장을 통째로 비교하는 대신 **성질**을 검사했습니다 — 1) 컨텍스트에 검색된 3개 이름이 모두 들어갔는가(근거를 제대로 엮었는가) 2) 답변이 비어 있지 않고 충분한 길이인가(호출이 성공했는가) 3) 답변에 검색된 상품 이름이 **적어도 하나** 나오는가(근거 밖 상품을 지어내지 않았는지의 약한 확인) 4) `messages` 에 `system` 규칙을 실제로 넣었는가.
- **흔한 실수**: 컨텍스트 없이 질문만 던지는 것입니다. 그러면 모델은 우리 쇼핑몰에 없는 상품을 그럴듯하게 지어냅니다(환각). **검색 결과를 프롬프트에 넣는 것**이 RAG 의 핵심입니다.
- **대안**: `n_results` 를 늘려 후보를 더 주거나, 컨텍스트에 `description` 까지 넣어 더 구체적인 추천을 받을 수 있습니다. 다만 프롬프트가 길어질수록 **비용과 지연**이 늘어납니다(문제 10 의 K 맞바꿈과 이어지는 이야기입니다).